Install libraries

In [11]:
!pip install langchain_unstructured -q
!pip install unstructured -q
!pip install datasets -q
!pip install ragas -q

import

In [50]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

import os
import pandas as pd
import requests
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser


load env

In [ ]:
os.environ["GOOGLE_API_KEY"] = "****"

define llm and embeddin model

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
vector = embeddings.embed_query("hello, world!")

model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-001",
    # model = "models/gemini-2.5.pro-exp-03-25",
    temperature=0,
    max_tokens=None,
    timeout=None,
    )

load and split data

In [2]:
# Load the data
loader = TextLoader('data.txt')
documents = loader.load()

# Chunk the data
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

Created a chunk of size 546, which is longer than the specified 500
Created a chunk of size 508, which is longer than the specified 500
Created a chunk of size 539, which is longer than the specified 500


retriever

In [49]:
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()

prompt and rag chain

In [10]:
# Define prompt template
template = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use two sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

# Setup RAG pipeline
rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()} 
    | prompt 
    | model
    | StrOutputParser() 
)

In [18]:
rag_chain.invoke("whats your contact number?")

'My contact number is +91 8553439205.'

RAGAs

In [36]:
from datasets import Dataset

questions = ["whats your contact number?", 
             "which companies have you worked?",
             "Hi",
            ]
ground_truths = ["My contact number is +91 8553439205",
                "I have worked in RGBSI pvt ltd, Jaivel Aerospace, Aivariant, and Codecraft Technologies pvt ltd. My roles included CNC Programmer and Data Scientist.",
                "How are you?"]
answers = []
contexts = []

# Inference
for query in questions:
  answers.append(rag_chain.invoke(query))
  contexts.append([docs.page_content for docs in retriever.get_relevant_documents(query)])

# To dict
data = {
  "user_input": questions,
  "response": answers,
  "retrieved_contexts": contexts,
  "reference": ground_truths
}

# Convert dict to dataset
dataset = Dataset.from_dict(data)

In [37]:
from ragas import evaluate
from ragas.metrics import (faithfulness, answer_relevancy, context_recall, context_precision,)

In [40]:
result = evaluate(
    llm = model,
    embeddings = embeddings,
    dataset = dataset, 
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
)

# df = result.to_pandas()

Evaluating:   8%|▊         | 1/12 [00:02<00:26,  2.41s/it]Retrying langchain_google_genai.chat_models._achat_with_retry.<locals>._achat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 8
}
].
Retrying langchain_google_genai.chat_models._achat_with_retry.<locals>._achat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You ex

In [51]:
df = result.to_pandas()
df

,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,whats your contact number?,[My information:\nmy name is Rohan shet\nmobil...,My contact number is +91 8553439205.,My contact number is +91 8553439205,1.0,1.0,1.000000,0.662184
1,which companies have you worked?,[NON IT Work Experience:\nI have worked in RGB...,"I have worked in RGBSI pvt ltd, Jaivel Aerospa...","I have worked in RGBSI pvt ltd, Jaivel Aerospa...",1.0,1.0,1.000000,0.689565
2,Hi,[My information:\nmy name is Rohan shet\nmobil...,"Hi, my name is Rohan shet and my contact numbe...",How are you?,0.0,0.0,0.666667,0.504625
